# Notebook 20 – Deep Learning Challenge

## Predict large retail orders
Use the local `data.csv` to predict whether a transaction has a large quantity. This is a new problem, different from the tumour classification workflow in Notebook 19.

## 1. Problem and dataset
**Definition:** Predict if an order line has at least five items.

**Example:** Quantity 8 is a large order (label 1); quantity 2 is not (label 0).

**Why it is used:** This is a small independent classification challenge using a real local dataset.

In [1]:
from pathlib import Path
import pandas as pd
data_path = Path('data.csv')
if not data_path.exists():
    data_path = Path(r'C:\Users\hemak\Sprint 9 – Deep Learning Fundamentals & Neural Networks\data.csv')
data = pd.read_csv(data_path, usecols=['Quantity', 'UnitPrice', 'InvoiceDate'], encoding='latin-1')
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], format='mixed', dayfirst=True, errors='coerce')
data = data.dropna().copy()
print(data.shape)

(541909, 3)


**Code Explanation:** The code reads the local CSV and converts invoice dates.

## 2. Input and target
**Definition:** Inputs are clues for the model; target is the answer it should predict.

**Example:** Inputs are unit price and time. Target is whether quantity is at least five. Quantity is not included as an input.

**Why it is used:** Leaving quantity out prevents the answer from being given directly to the model.

In [2]:
data['large_order'] = (data['Quantity'].abs() >= 5).astype(int)
data['hour'] = data['InvoiceDate'].dt.hour
data['weekday'] = data['InvoiceDate'].dt.dayofweek
X = data[['UnitPrice', 'hour', 'weekday']]
y = data['large_order']
print('Large-order share:', round(y.mean(), 2))

Large-order share: 0.42


**Code Explanation:** Quantity creates the label; price and time are the model inputs.

## 3. Basic EDA
**Definition:** EDA uses simple counts and summaries to understand the data.

**Example:** Count small and large order lines.

**Why it is used:** The class counts help us notice if one label is rare.

In [3]:
print(y.value_counts())
print(data[['UnitPrice', 'hour', 'weekday']].describe().round(1))

large_order
0    312362
1    229547
Name: count, dtype: int64
       UnitPrice      hour   weekday
count   541909.0  541909.0  541909.0
mean         4.6      13.1       2.7
std         96.8       2.4       1.9
min     -11062.1       6.0       0.0
25%          1.2      11.0       1.0
50%          2.1      13.0       3.0
75%          4.1      15.0       4.0
max      38970.0      20.0       6.0


**Code Explanation:** Counts show how many rows belong to each class.

## 4. Split and prepare
**Definition:** Split data before fitting the scaler or model.

**Example:** Keep 20% for the final test.

**Why it is used:** The test set remains unseen until evaluation.

In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
if len(data) > 15000:
    data = data.sample(15000, random_state=20)
X = data[['UnitPrice', 'hour', 'weekday']]
y = data['large_order']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=20)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=20)
scaler = StandardScaler().fit(X_train)
X_train = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
X_val = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
print('Train / validation / test:', len(X_train), len(X_val), len(X_test))

Train / validation / test: 10500 2250 2250


**Code Explanation:** The code splits the data and scales values using training rows only.

## 5. Architecture
**Definition:** An architecture is the arrangement of network layers.

**Example:** Three inputs → 16 hidden units → one output.

**Why it is used:** A small network fits this small tabular problem and is easy to compare.

In [6]:
import torch
from torch import nn
def make_model(hidden, dropout=0):
    return nn.Sequential(nn.Linear(3, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
print(make_model(16))

Sequential(
  (0): Linear(in_features=3, out_features=16, bias=True)
  (1): ReLU()
  (2): Dropout(p=0, inplace=False)
  (3): Linear(in_features=16, out_features=1, bias=True)
)


**Code Explanation:** The function builds a small network with three inputs and one output.

## 6. Train two configurations
**Definition:** Compare two settings using the same training and test split.

**Example:** Try 16 units, then 32 units with dropout.

**Why it is used:** Validation data tells us whether the change helped.

In [7]:
from sklearn.metrics import f1_score
def train_model(model, epochs=4):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            loss_fn(model(batch_x), batch_y).backward()
            optimizer.step()
    return model
model_a = train_model(make_model(16))
model_b = train_model(make_model(32, dropout=0.2))
print('Both models trained.')

Both models trained.


**Code Explanation:** The function trains two neural networks with different sizes.

## 7. Evaluate and compare
**Definition:** Evaluation compares model predictions with known labels.

**Example:** F1 combines precision and recall.

**Why it is used:** Compare configurations on validation data; save test data for the final check.

In [9]:
def get_predictions(model, features):
    model.eval()
    with torch.no_grad():
        return (torch.sigmoid(model(features)) >= 0.5).int().numpy().ravel()
pred_a = get_predictions(model_a, X_val)
pred_b = get_predictions(model_b, X_val)
score_a = f1_score(y_val.numpy().ravel(), pred_a)
score_b = f1_score(y_val.numpy().ravel(), pred_b)
print('16 units validation F1:', round(score_a, 3))
print('32 units + dropout validation F1:', round(score_b, 3))

16 units validation F1: 0.413
32 units + dropout validation F1: 0.408


**Code Explanation:** The models make validation predictions; F1 scores compare them.

## 8. Error analysis
**Definition:** Error analysis looks at predictions that do not match the answer.

**Example:** A false positive is a small order predicted as large.

**Why it is used:** Mistakes can show which features or thresholds need improvement.

In [10]:
from sklearn.metrics import confusion_matrix
best_pred = pred_b if score_b >= score_a else pred_a
print(confusion_matrix(y_val.numpy().ravel(), best_pred))

[[1048  250]
 [ 639  313]]


**Code Explanation:** The confusion matrix summarizes validation mistakes.

## 9. Select the final model
Choose the configuration with the higher F1 score. A larger network is not automatically better.

In [11]:
final_model = model_b if score_b >= score_a else model_a
print('Selected:', '32 units + dropout' if score_b >= score_a else '16 units')

Selected: 16 units


**Code Explanation:** The model with the better validation F1 is selected.

## 10. Final evaluation

Check the selected configuration on the test set once.

In [12]:
from sklearn.metrics import classification_report
final_model = model_b if score_b >= score_a else model_a
test_pred = get_predictions(final_model, X_test)
print(classification_report(y_test.numpy().ravel(), test_pred, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.62      0.82      0.71      1297
         1.0       0.56      0.31      0.40       953

    accuracy                           0.61      2250
   macro avg       0.59      0.57      0.55      2250
weighted avg       0.59      0.61      0.58      2250



**Code Explanation:** The report evaluates the selected model on the test data.

## 11. Limitations and future improvements

This small demo uses only price and time, so it may not predict order size well. The five-item cutoff is a teaching choice. Add product features, test on a later time period, and choose a decision threshold based on error costs.